# Olist Customer Growth Intelligence — v3: order-risk models ("did this order go wrong?")

**Run on Kaggle:** *Add Input* → "Brazilian E-Commerce Public Dataset by Olist" → **Run All**.
Outputs in `/kaggle/working`: `order_risk_model.joblib`, `orders_scored.csv`.
v2 (`retention_model.joblib`) is untouched; v3 adds a second, separate model.

**Why v3:** a 180-day repeat happens for <2% of customers (902 events), and v2 showed no out-of-time ranking power.
v3 predicts something that is far more frequent and actionable *before* the customer is lost:

| target | definition |
|---|---|
| `late` | delivered after the promised (estimated) date, or never delivered once the promise date passed |
| `bad_review` | lowest review score on the order ≤ 2 |
| `canceled` | status canceled or unavailable |
| `went_wrong` | any of the three (composite **event**, not a weighted score) |

**Scoring time = the moment of purchase.** Only facts known then are features: basket, product size, price/freight,
payment, customer & seller location (distance from `geolocation`), the promised delivery date, and the seller's **own
track record up to that moment** (point-in-time, checked for leakage below).

**Validation:** gapped out-of-time split (train on orders whose outcomes were resolved before the test period starts).
Everything printed below is computed by this run; nothing is hard-coded.

In [1]:
import os, json, math, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
try:
    display
except NameError:          # running outside Jupyter
    display = print

SEED = 42
WINDOW_DAYS, MIN_GAP_DAYS = 180, 1.0            # same repeat definition as v1/v2 (used in section 4)
RESOLVE_DAYS = 60                                # an order's outcome is treated as known 60 days after purchase
TEST_START = pd.Timestamp("2018-03-01")          # out-of-time test period starts here
TRAIN_END_GAPPED = TEST_START - pd.Timedelta(days=RESOLVE_DAYS)
TARGETS = ["went_wrong", "late", "bad_review", "canceled"]
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def find_data_dir():
    cands = [Path(os.environ["OLIST_DIR"])] if os.environ.get("OLIST_DIR") else []
    for root, _, files in os.walk("/kaggle/input"):
        if "olist_orders_dataset.csv" in files:
            cands.append(Path(root)); break
    cands.append(Path("."))
    for c in cands:
        if (c / "olist_orders_dataset.csv").exists():
            return c
    raise FileNotFoundError("Attach the Olist dataset as a notebook input (or set OLIST_DIR).")

DATA_DIR = find_data_dir()
print("Data directory:", DATA_DIR, "| Output directory:", OUT_DIR)

def wilson(k, n, z=1.959964):
    """Wilson 95% interval in percent (no statsmodels needed)."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (100 * max(0.0, c - h), 100 * min(1.0, c + h))

def rate_table(df, by, target):
    t = df.groupby(by, observed=True)[target].agg(orders="size", events="sum").reset_index()
    t["rate_pct"] = (t.events / t.orders * 100).round(2)
    ci = [wilson(k, n) for k, n in zip(t.events, t.orders)]
    t["ci_low"] = [round(c[0], 2) for c in ci]; t["ci_high"] = [round(c[1], 2) for c in ci]
    return t.rename(columns={by: "band"})

def attach(left, right, on="order_id"):
    """Left-join that refuses overlapping columns and row multiplication (grain guard)."""
    dup = [c for c in right.columns if c in left.columns and c != on]
    assert not dup, f"overlapping columns {dup}"
    out = left.merge(right, on=on, how="left", validate="many_to_one")
    assert len(out) == len(left), "join changed row count"
    return out

Data directory: /kaggle/input/datasets/olistbr/brazilian-ecommerce | Output directory: /kaggle/working


In [2]:
rd = lambda name: pd.read_csv(DATA_DIR / name)
customers, orders = rd("olist_customers_dataset.csv"), rd("olist_orders_dataset.csv")
items, pay, reviews = rd("olist_order_items_dataset.csv"), rd("olist_order_payments_dataset.csv"), rd("olist_order_reviews_dataset.csv")
products, sellers, geo = rd("olist_products_dataset.csv"), rd("olist_sellers_dataset.csv"), rd("olist_geolocation_dataset.csv")
try:
    trans = rd("product_category_name_translation.csv")
except FileNotFoundError:
    trans = None

for c in ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
          "order_delivered_customer_date", "order_estimated_delivery_date"]:
    orders[c] = pd.to_datetime(orders[c])
for c in ["review_creation_date", "review_answer_timestamp"]:
    reviews[c] = pd.to_datetime(reviews[c])

orders = orders.merge(customers[["customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_state"]],
                      on="customer_id", how="left", validate="many_to_one")
assert orders.order_id.is_unique and orders.customer_unique_id.notna().all()
orders["is_value_order"] = ~orders.order_status.isin(["canceled", "unavailable"])
OBS_END = orders.loc[orders.is_value_order, "order_purchase_timestamp"].max()   # same primary definition as v2
print(f"Orders: {len(orders):,} | customers: {orders.customer_unique_id.nunique():,} | sellers: {len(sellers):,} | "
      f"geolocation rows: {len(geo):,} | observation end (last value order): {OBS_END}")

Orders: 99,441 | customers: 96,096 | sellers: 3,095 | geolocation rows: 1,000,163 | observation end (last value order): 2018-09-03 09:06:57


## 1. Order outcomes and the resolved population

In [3]:
o = orders.copy()
o["canceled"] = (~o.is_value_order).astype(int)
deliv, est = o.order_delivered_customer_date, o.order_estimated_delivery_date
o["late"] = np.where(o.canceled == 1, np.nan,                                   # not defined for canceled orders
            np.where(deliv.notna(), (deliv.dt.normalize() > est).astype(float),  # delivered after the promised day
            np.where((est < OBS_END) & (o.order_status != "delivered"), 1.0, np.nan)))  # never delivered and promise passed

rv = reviews.groupby("order_id").agg(min_review=("review_score", "min"), review_answered=("review_answer_timestamp", "max")).reset_index()
o = attach(o, rv)
o["has_review"] = o.min_review.notna().astype(int)
o["bad_review"] = (o.min_review <= 2).astype(int)
o["went_wrong"] = ((o.canceled == 1) | (o.late == 1) | (o.bad_review == 1)).astype(int)

n_items_per_order = items.groupby("order_id").size()
o["has_items"] = o.order_id.isin(n_items_per_order.index)
o["resolved"] = o.order_purchase_timestamp <= OBS_END - pd.Timedelta(days=RESOLVE_DAYS)
o["in_population"] = o.resolved & o.has_items & (o.late.notna() | (o.canceled == 1))

no_items = o[~o.has_items]
print(f"Orders with NO item rows: {len(no_items):,}, of which canceled/unavailable: {no_items.canceled.mean()*100:.1f}%.")
print("  -> excluded: at purchase time every order has items, so 'no items' is a data artifact that would leak the outcome.")

pop = o[o.in_population]
summ = []
for t in TARGETS:
    s = pop[t].dropna(); k, n = int(s.sum()), len(s); lo, hi = wilson(k, n)
    summ.append(dict(target=t, orders=n, events=k, rate_pct=round(k / n * 100, 2), ci_low=round(lo, 2), ci_high=round(hi, 2)))
summ = pd.DataFrame(summ)
print(f"\nResolved population: {len(pop):,} orders purchased up to {(OBS_END - pd.Timedelta(days=RESOLVE_DAYS)).date()}")
display(summ)
print("Overlap of the components inside 'went_wrong' (rows: late, columns: bad review; canceled shown separately):")
display(pd.crosstab(pop.late.map({0.0: "on time", 1.0: "late"}).fillna("canceled"), pop.bad_review.map({0: "review ok/none", 1: "bad review"}), margins=True))
print(f"Orders without any review: {(1 - pop.has_review.mean())*100:.1f}% (counted as 'not a bad review').")

Orders with NO item rows: 775, of which canceled/unavailable: 99.0%.
  -> excluded: at purchase time every order has items, so 'no items' is a data artifact that would leak the outcome.

Resolved population: 86,790 orders purchased up to 2018-07-05


,target,orders,events,rate_pct,ci_low,ci_high
0,went_wrong,86790,15375,17.72,17.46,17.97
1,late,86389,7525,8.71,8.52,8.90
2,bad_review,86790,12649,14.57,14.34,14.81
3,canceled,86790,401,0.46,0.42,0.51


Overlap of the components inside 'went_wrong' (rows: late, columns: bad review; canceled shown separately):


bad_review,bad review,review ok/none,All
late,,,
canceled,318,83,401
late,4882,2643,7525
on time,7449,71415,78864
All,12649,74141,86790


Orders without any review: 0.8% (counted as 'not a bad review').


## 2. Purchase-time inputs and point-in-time seller history

Each order is attributed to its **primary seller** (the seller of its most expensive item). A seller's history only
counts outcomes that were **known before this order was placed**:
late/on-time is known at delivery (or at the promise date if never delivered), a bad review when the review was answered,
and a cancellation conservatively at the promised date. Rates are shrunk toward the global rate (20 pseudo-orders) so
new sellers aren't scored on 1-2 orders. Global rates use only training-period information.

In [4]:
it = items.merge(products[["product_id", "product_category_name", "product_weight_g", "product_length_cm",
                            "product_height_cm", "product_width_cm"]], on="product_id", how="left", validate="many_to_one")
it["vol_cm3"] = it.product_length_cm * it.product_height_cm * it.product_width_cm
if trans is not None:
    it = it.merge(trans, on="product_category_name", how="left", validate="many_to_one")
    it["category"] = it.product_category_name_english.fillna(it.product_category_name)
else:
    it["category"] = it.product_category_name
agg = it.groupby("order_id").agg(n_items=("order_item_id", "count"), n_products=("product_id", "nunique"),
                                 n_sellers=("seller_id", "nunique"), total_price=("price", "sum"), total_freight=("freight_value", "sum"),
                                 total_weight_g=("product_weight_g", lambda s: s.sum(min_count=len(s))),
                                 max_weight_g=("product_weight_g", "max"),
                                 total_volume_cm3=("vol_cm3", lambda s: s.sum(min_count=len(s)))).reset_index()
prim = it.sort_values(["order_id", "price", "order_item_id"], ascending=[True, False, True]).drop_duplicates("order_id")[["order_id", "seller_id", "category"]]
prim = prim.merge(sellers.rename(columns={"seller_zip_code_prefix": "seller_zip"})[["seller_id", "seller_zip", "seller_state"]],
                  on="seller_id", how="left", validate="many_to_one")
pt = pay.sort_values(["order_id", "payment_value"], ascending=[True, False]).groupby("order_id").agg(
    payment_type=("payment_type", "first"), installments=("payment_installments", "max")).reset_index()

o = attach(attach(attach(o, agg), prim), pt)
o = o.sort_values(["order_purchase_timestamp", "order_id"]).reset_index(drop=True)
o["customer_prior_orders"] = o.groupby("customer_unique_id").cumcount()      # orders placed earlier by the same person

# zip-prefix centroids (drop coordinates outside Brazil's bounding box)
g = geo[geo.geolocation_lat.between(-34, 6) & geo.geolocation_lng.between(-74, -34)]
zc = g.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]].mean()
ZIP_CENTROIDS = {int(z): (float(a), float(b)) for z, (a, b) in zc.iterrows()}

# --- point-in-time seller history -------------------------------------------------------------
os_ = o[o.seller_id.notna()]
EVENTS = {
    "orders": os_.assign(known_at=os_.order_purchase_timestamp, value=1),
    "late":   os_[os_.late.notna()].assign(known_at=lambda d: d.order_delivered_customer_date.fillna(d.order_estimated_delivery_date), value=lambda d: d.late),
    "bad":    os_[os_.has_review == 1].assign(known_at=lambda d: d.review_answered, value=lambda d: d.bad_review),
    "cancel": os_.assign(known_at=os_.order_estimated_delivery_date, value=os_.canceled),
}
EVENTS = {k: v[["order_id", "seller_id", "known_at", "value"]].dropna(subset=["known_at"]).sort_values("known_at") for k, v in EVENTS.items()}

def prior_counts(ev, query, ts_col, prefix):
    """For each query row: number (n) and sum (k) of the seller's events known STRICTLY before query[ts_col]."""
    e = ev.copy(); e["known_at"] = pd.to_datetime(e["known_at"]).astype("datetime64[ns]")
    e["_n"] = e.groupby("seller_id").cumcount() + 1
    e["_k"] = e.groupby("seller_id")["value"].cumsum()
    q = query[["order_id", "seller_id", ts_col]].copy(); q[ts_col] = pd.to_datetime(q[ts_col]).astype("datetime64[ns]")
    q = q.sort_values(ts_col)
    m = pd.merge_asof(q, e[["seller_id", "known_at", "_n", "_k"]], left_on=ts_col, right_on="known_at",
                      by="seller_id", allow_exact_matches=False, direction="backward")
    return m[["order_id", "_n", "_k"]].rename(columns={"_n": f"{prefix}_n", "_k": f"{prefix}_k"}).fillna(0)

hist = os_[["order_id"]]
for name, prefix in [("orders", "seller_orders"), ("late", "seller_late"), ("bad", "seller_bad"), ("cancel", "seller_cancel")]:
    hist = hist.merge(prior_counts(EVENTS[name], os_, "order_purchase_timestamp", prefix), on="order_id", how="left", validate="one_to_one")
hist = hist.rename(columns={"seller_orders_n": "seller_prior_orders"}).drop(columns="seller_orders_k")
o = attach(o, hist)

GLOBAL_RATES = {k: float(EVENTS[e].loc[EVENTS[e].known_at < TRAIN_END_GAPPED, "value"].mean())
                for k, e in [("late", "late"), ("bad_review", "bad"), ("cancel", "cancel")]}
print("Global rates used for shrinkage (training period only):", {k: round(v, 4) for k, v in GLOBAL_RATES.items()})

Global rates used for shrinkage (training period only): {'late': 0.0631, 'bad_review': 0.1319, 'cancel': 0.0055}


### 2a. Leakage check: recompute seller history by brute force for 300 random orders

In [5]:
rng = np.random.default_rng(SEED)
sample = o[o.seller_id.notna()].iloc[rng.choice(int(o.seller_id.notna().sum()), 300, replace=False)]
bad_rows = 0
for _, r in sample.iterrows():
    for name, prefix in [("late", "seller_late"), ("bad", "seller_bad"), ("cancel", "seller_cancel")]:
        ev = EVENTS[name]; s = ev[(ev.seller_id == r.seller_id) & (ev.known_at < r.order_purchase_timestamp)]
        if len(s) != r[f"{prefix}_n"] or s.value.sum() != r[f"{prefix}_k"]:
            bad_rows += 1
    ev = EVENTS["orders"]
    if ((ev.seller_id == r.seller_id) & (ev.known_at < r.order_purchase_timestamp)).sum() != r.seller_prior_orders:
        bad_rows += 1
    if r.order_id in set(EVENTS["late"].loc[(EVENTS["late"].seller_id == r.seller_id) & (EVENTS["late"].known_at < r.order_purchase_timestamp), "order_id"]):
        bad_rows += 1   # an order must never see its own outcome
assert bad_rows == 0, f"{bad_rows} mismatches - seller history is not point-in-time"
print("Leakage check passed: 300/300 orders match a brute-force recomputation, and no order sees its own outcome.")

Leakage check passed: 300/300 orders match a brute-force recomputation, and no order sees its own outcome.


### 2b. Shared feature function (identical to `order_features.py` used by the app)

In [6]:
# order_features.py - ONE feature function shared by the v3 notebook (training) and the app (scoring).
# The notebook embeds this exact file as a cell, so training and serving cannot drift apart.
import math
import numpy as np
import pandas as pd

NUM = ["n_items", "n_products", "n_sellers", "log_price", "log_freight", "freight_share",
       "total_weight_kg", "max_weight_kg", "total_volume_l", "dims_missing",
       "distance_km", "same_state", "promised_days", "installments",
       "purchase_hour", "purchase_dow", "is_holiday_season",
       "seller_prior_orders_log", "seller_has_history", "seller_prior_late_rate",
       "seller_prior_bad_review_rate", "seller_prior_cancel_rate", "customer_prior_orders"]
CAT = ["customer_state", "seller_state", "category", "payment_type"]
FEATURES = NUM + CAT
SHRINK_A = 20.0   # pseudo-orders used to shrink a seller's rates toward the global rate


def haversine_km(lat1, lng1, lat2, lng2):
    if any(v is None or (isinstance(v, float) and math.isnan(v)) for v in (lat1, lng1, lat2, lng2)):
        return float("nan")
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lng2 - lng1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * 6371.0 * math.asin(math.sqrt(a))


def _num(x):
    try:
        x = float(x)
    except (TypeError, ValueError):
        return float("nan")
    return x


def shrunk_rate(k, n, prior, a=SHRINK_A):
    k, n = _num(k), _num(n)
    k = 0.0 if math.isnan(k) else k
    n = 0.0 if math.isnan(n) else n
    return (k + a * prior) / (n + a)


def features_from_inputs(o, zip_centroids, global_rates):
    """o: dict of raw order inputs known at purchase time (see keys below).
    zip_centroids: {zip_prefix(int): (lat, lng)}. global_rates: {"late":..,"bad_review":..,"cancel":..}.
    Seller history (seller_prior_*) must be computed as of the purchase moment by the caller."""
    def zc(z):
        try:
            return zip_centroids.get(int(z), (float("nan"), float("nan")))
        except (TypeError, ValueError):
            return (float("nan"), float("nan"))

    clat, clng = zc(o.get("customer_zip"))
    slat, slng = zc(o.get("seller_zip"))
    ts = pd.Timestamp(o["purchase_ts"])
    est = o.get("estimated_delivery_date")
    promised = (pd.Timestamp(est) - ts).total_seconds() / 86400 if est is not None and not pd.isna(est) else float("nan")
    price, freight = _num(o.get("total_price")), _num(o.get("total_freight"))
    den = price + freight
    vol = _num(o.get("total_volume_cm3"))
    n_prior = _num(o.get("seller_prior_orders")); n_prior = 0.0 if math.isnan(n_prior) else n_prior
    cs, ss = str(o.get("customer_state") or "missing"), str(o.get("seller_state") or "missing")
    f = {
        "n_items": min(_num(o.get("n_items")), 10),
        "n_products": min(_num(o.get("n_products")), 10),
        "n_sellers": min(_num(o.get("n_sellers")), 5),
        "log_price": math.log1p(max(price, 0)) if not math.isnan(price) else float("nan"),
        "log_freight": math.log1p(max(freight, 0)) if not math.isnan(freight) else float("nan"),
        "freight_share": freight / den if den and den > 0 else float("nan"),
        "total_weight_kg": _num(o.get("total_weight_g")) / 1000,
        "max_weight_kg": _num(o.get("max_weight_g")) / 1000,
        "total_volume_l": vol / 1000,
        "dims_missing": float(math.isnan(vol)),
        "distance_km": min(haversine_km(clat, clng, slat, slng), 4500),
        "same_state": float(cs == ss and cs != "missing"),
        "promised_days": min(max(promised, 0), 90) if not math.isnan(promised) else float("nan"),
        "installments": min(_num(o.get("installments")), 24),
        "purchase_hour": float(ts.hour),
        "purchase_dow": float(ts.dayofweek),
        "is_holiday_season": float(ts.month in (11, 12)),
        "seller_prior_orders_log": math.log1p(n_prior),
        "seller_has_history": float(n_prior >= 5),
        "seller_prior_late_rate": shrunk_rate(o.get("seller_late_k"), o.get("seller_late_n"), global_rates["late"]),
        "seller_prior_bad_review_rate": shrunk_rate(o.get("seller_bad_k"), o.get("seller_bad_n"), global_rates["bad_review"]),
        "seller_prior_cancel_rate": shrunk_rate(o.get("seller_cancel_k"), o.get("seller_cancel_n"), global_rates["cancel"]),
        "customer_prior_orders": min(_num(o.get("customer_prior_orders")) if not math.isnan(_num(o.get("customer_prior_orders"))) else 0.0, 10),
        "customer_state": cs, "seller_state": ss,
        "category": str(o.get("category") or "missing"),
        "payment_type": str(o.get("payment_type") or "missing"),
    }
    return f


def build_frame(rows, zip_centroids, global_rates):
    """Apply features_from_inputs to many orders (list of dicts or a DataFrame) -> DataFrame[FEATURES]."""
    if isinstance(rows, pd.DataFrame):
        rows = rows.to_dict("records")
    return pd.DataFrame([features_from_inputs(r, zip_centroids, global_rates) for r in rows], columns=FEATURES)

In [7]:
raw = o.rename(columns={"customer_zip_code_prefix": "customer_zip", "order_purchase_timestamp": "purchase_ts",
                        "order_estimated_delivery_date": "estimated_delivery_date"})
INPUT_COLS = ["customer_zip", "customer_state", "seller_zip", "seller_state", "purchase_ts", "estimated_delivery_date",
              "n_items", "n_products", "n_sellers", "total_price", "total_freight", "total_weight_g", "max_weight_g", "total_volume_cm3",
              "category", "payment_type", "installments", "seller_prior_orders", "seller_late_k", "seller_late_n",
              "seller_bad_k", "seller_bad_n", "seller_cancel_k", "seller_cancel_n", "customer_prior_orders"]
X_all = build_frame(raw[INPUT_COLS], ZIP_CENTROIDS, GLOBAL_RATES)
df = pd.concat([o[["order_id", "customer_unique_id", "seller_id", "order_purchase_timestamp", "order_status", "in_population"] + TARGETS].reset_index(drop=True),
                X_all.reset_index(drop=True)], axis=1)
assert len(df) == len(o) and df.order_id.is_unique
print(f"Feature frame: {len(df):,} orders x {len(FEATURES)} features | missing distance: {df.distance_km.isna().mean()*100:.1f}%")
display(df.loc[df.in_population, NUM].describe().T[["mean", "50%", "min", "max"]].round(2))

Feature frame: 99,441 orders x 27 features | missing distance: 1.3%


,mean,50%,min,max
n_items,1.14,1.00,1.00,10.00
n_products,1.04,1.00,1.00,7.00
n_sellers,1.01,1.00,1.00,5.00
log_price,4.46,4.48,0.62,9.51
log_freight,2.98,2.89,0.00,7.49
freight_share,0.21,0.18,0.00,0.96
total_weight_kg,2.45,0.80,0.00,184.40
max_weight_kg,2.17,0.70,0.00,40.42
total_volume_l,17.80,7.60,0.17,1476.00
dims_missing,0.00,0.00,0.00,1.00


## 3. Which purchase-time facts are associated with an order going wrong? (Wilson 95% CIs)

In [8]:
c = df[df.in_population].copy()
c["Distance"] = pd.cut(c.distance_km, [-1, 100, 500, 1000, np.inf], labels=["<100 km", "100-500", "500-1000", ">1000 km"])
c["Promised days"] = pd.cut(c.promised_days, [-1, 14, 21, 30, np.inf], labels=["<=14d", "15-21d", "22-30d", ">30d"])
c["Same state"] = c.same_state.map({1.0: "same", 0.0: "different"})
c["Seller history"] = np.where(c.seller_has_history == 0, "new seller (<5 orders)",
                       pd.cut(c.seller_prior_late_rate, [-1, .05, .10, .20, 1], labels=["late <=5%", "5-10%", "10-20%", ">20%"]).astype(str))
c["Total weight"] = pd.cut(c.total_weight_kg, [-1, 1, 5, 15, np.inf], labels=["<1 kg", "1-5", "5-15", ">15 kg"])
c["Holiday season"] = c.is_holiday_season.map({1.0: "Nov-Dec", 0.0: "other"})
c["Sellers in order"] = np.where(c.n_sellers >= 2, "2+", "1")
dims = ["Distance", "Promised days", "Same state", "Seller history", "Total weight", "Holiday season", "Sellers in order"]
assoc = pd.concat([rate_table(c, d, "went_wrong").assign(dimension=d) for d in dims], ignore_index=True)
display(assoc[["dimension", "band", "orders", "events", "rate_pct", "ci_low", "ci_high"]])

,dimension,band,orders,events,rate_pct,ci_low,ci_high
0,Distance,<100 km,15573,2185,14.03,13.49,14.59
1,Distance,100-500,33372,5688,17.04,16.64,17.45
2,Distance,500-1000,23435,4298,18.34,17.85,18.84
3,Distance,>1000 km,13969,3127,22.39,21.70,23.08
4,Promised days,<=14d,7435,1010,13.58,12.82,14.38
5,Promised days,15-21d,20225,3559,17.60,17.08,18.13
6,Promised days,22-30d,39661,7210,18.18,17.80,18.56
7,Promised days,>30d,19469,3596,18.47,17.93,19.02
8,Same state,different,56243,11034,19.62,19.29,19.95
9,Same state,same,30547,4341,14.21,13.82,14.61


## 4. Link back to retention: do customers whose FIRST order went wrong come back less?
Same repeat definition and eligibility as v1 (occasion rule ≥ 1 day, 180 days, observation end = last value order),
so the eligible cohort should reproduce v1's 58,101 customers / 1.95% on the real data. Observational only.

In [9]:
vo = o[o.is_value_order].sort_values(["customer_unique_id", "order_purchase_timestamp", "order_id"])
gg = vo.groupby("customer_unique_id")["order_purchase_timestamp"]
vo = vo.assign(days_since_first=(vo.order_purchase_timestamp - gg.transform("min")).dt.total_seconds() / 86400)
first = vo.drop_duplicates("customer_unique_id").copy()
nxt = vo[vo.days_since_first >= MIN_GAP_DAYS].groupby("customer_unique_id")["days_since_first"].min().rename("days_to_next_occasion").reset_index()
first = first.merge(nxt, on="customer_unique_id", how="left", validate="one_to_one")
first["repeat_180d"] = first.days_to_next_occasion.le(WINDOW_DAYS).astype(int)
fc = first[first.order_purchase_timestamp <= OBS_END - pd.Timedelta(days=WINDOW_DAYS)].copy()
print(f"Eligible first-order cohort: {len(fc):,} customers | 180-day repeat {fc.repeat_180d.mean()*100:.2f}% "
      "(v1 real-data run: 58,101 / 1.95%)")

rows = []
for flag, label in [("went_wrong", "First order went wrong (late or bad review)"), ("late", "First order late"), ("bad_review", "First order bad review")]:
    for v, name in [(1, "yes"), (0, "no")]:
        s = fc[fc[flag] == v]; k, n = int(s.repeat_180d.sum()), len(s); lo, hi = wilson(k, n)
        rows.append(dict(first_order=label, value=name, customers=n, repeaters=k, repeat_pct=round(k / max(n, 1) * 100, 2), ci_low=round(lo, 2), ci_high=round(hi, 2)))
link = pd.DataFrame(rows)
display(link)
ww = link[link.first_order.str.startswith("First order went")].set_index("value")
print(f"Repeat rate when the first order went wrong: {ww.loc['yes','repeat_pct']}% vs {ww.loc['no','repeat_pct']}% when it didn't "
      f"(ratio {ww.loc['yes','repeat_pct']/ww.loc['no','repeat_pct']:.2f}). Canceled first orders are not value orders, so they are not in this cohort.")

Eligible first-order cohort: 58,101 customers | 180-day repeat 1.95% (v1 real-data run: 58,101 / 1.95%)


,first_order,value,customers,repeaters,repeat_pct,ci_low,ci_high
0,First order went wrong (late or bad review),yes,10271,173,1.68,1.45,1.95
1,First order went wrong (late or bad review),no,47830,959,2.01,1.88,2.13
2,First order late,yes,5235,84,1.60,1.30,1.98
3,First order late,no,52864,1048,1.98,1.87,2.10
4,First order bad review,yes,8509,139,1.63,1.39,1.93
5,First order bad review,no,49592,993,2.00,1.88,2.13


Repeat rate when the first order went wrong: 1.68% vs 2.01% when it didn't (ratio 0.84). Canceled first orders are not value orders, so they are not in this cohort.


## 5. Models: logistic regression vs gradient boosting, per target
Primary evaluation is **gapped out-of-time**: train on orders purchased before TRAIN_END_GAPPED (= test start − 60 days) (outcomes resolved), test on orders
from TEST_START (2018-03-01). Ungapped results are shown for reference. A gradient-boosting model is deployed for a target only if its
out-of-time AUC lower bound beats the logistic model's point estimate (same rule as v2).

In [10]:
import joblib, sklearn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def make_logit():
    # median-impute, standardise, then clip to +-5 SD so a rare value (e.g. unknown zip) can't produce an extreme score
    pre = ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                                               ("clip", FunctionTransformer(np.clip, kw_args={"a_min": -5, "a_max": 5}))]), NUM),
                             ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), CAT)])
    return Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, C=0.5, random_state=SEED))])

def make_hgb():
    pre = ColumnTransformer([("num", "passthrough", NUM),
                             ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT)])
    cat_idx = list(range(len(NUM), len(NUM) + len(CAT)))
    return Pipeline([("pre", pre), ("clf", HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, l2_regularization=1.0,
                                                                          min_samples_leaf=100, categorical_features=cat_idx, random_state=SEED))])
FACTORIES = {"logistic": make_logit, "gradient_boosting": make_hgb}

def boot_auc(y, p, n=200):
    r = np.random.default_rng(SEED); idx = np.arange(len(y)); out = []
    for _ in range(n):
        s = r.choice(idx, len(idx))
        if y[s].min() != y[s].max():
            out.append(roc_auc_score(y[s], p[s]))
    return [float(x) for x in np.percentile(out, [2.5, 97.5])]

def deciles(y, p):
    d = pd.DataFrame({"y": y, "p": p}); d["decile"] = 10 - pd.qcut(d.p.rank(method="first"), 10, labels=False)
    t = d.groupby("decile").agg(orders=("y", "size"), events=("y", "sum"), mean_pred_pct=("p", lambda s: s.mean() * 100)).reset_index()
    t["actual_pct"] = (t.events / t.orders * 100).round(2); t["mean_pred_pct"] = t.mean_pred_pct.round(2)
    t["lift"] = (t.actual_pct / (y.mean() * 100)).round(2)
    return t

data = {t: df[df.in_population & df[t].notna()].copy() for t in TARGETS}
BASELINE_FEATS = {"distance_km": 1, "promised_days": 1, "seller_prior_late_rate": 1, "seller_prior_bad_review_rate": 1}

def evaluate(target, model, gapped=True):
    d = data[target]; y_all = d[target].astype(int)
    tr_mask = d.order_purchase_timestamp < (TRAIN_END_GAPPED if gapped else TEST_START)
    te_mask = d.order_purchase_timestamp >= TEST_START
    pipe = FACTORIES[model]().fit(d.loc[tr_mask, FEATURES], y_all[tr_mask])
    p_tr = pipe.predict_proba(d.loc[tr_mask, FEATURES])[:, 1]
    p = pipe.predict_proba(d.loc[te_mask, FEATURES])[:, 1]; y = y_all[te_mask].values
    singles = {f: float(max(a, 1 - a)) for f in BASELINE_FEATS
               for a in [roc_auc_score(y, d.loc[te_mask, f].fillna(d.loc[tr_mask, f].median()))]}   # direction-free
    return dict(target=target, model=model, split="gapped" if gapped else "ungapped", n_train=int(tr_mask.sum()), events_train=int(y_all[tr_mask].sum()),
                n_test=int(te_mask.sum()), events_test=int(y.sum()), base_rate_test_pct=round(y.mean() * 100, 2),
                mean_pred_test_pct=round(p.mean() * 100, 2), roc_auc=float(roc_auc_score(y, p)), roc_auc_ci=boot_auc(y, p),
                pr_auc=float(average_precision_score(y, p)), single_feature_auc=singles, deciles=deciles(y, p),
                pipe=pipe, p_train=p_tr, p_test=p, y_test=y)

RESULTS = {}
for t in TARGETS:
    for m in FACTORIES:
        RESULTS[(t, m, "gapped")] = evaluate(t, m, True)
    RESULTS[(t, "logistic", "ungapped")] = evaluate(t, "logistic", False)

perf = pd.DataFrame([{k: v for k, v in r.items() if k in ("target", "model", "split", "n_train", "events_train", "n_test", "events_test",
                                                          "base_rate_test_pct", "mean_pred_test_pct", "roc_auc", "pr_auc")}
                     | {"auc_ci": f"{r['roc_auc_ci'][0]:.3f}-{r['roc_auc_ci'][1]:.3f}", "top_decile_lift": r["deciles"].lift.iloc[0]}
                     for r in RESULTS.values()])
perf["roc_auc"] = perf.roc_auc.round(3); perf["pr_auc"] = perf.pr_auc.round(4)
print("Out-of-time performance (test = orders from", TEST_START.date(), "on). Random ROC-AUC = 0.5; random PR-AUC = base rate.")
display(perf)
print("Single-feature AUCs on the gapped test set (a model should beat these to be worth having):")
display(pd.DataFrame({t: RESULTS[(t, "logistic", "gapped")]["single_feature_auc"] for t in TARGETS}).round(3))

DEPLOY = {}
for t in TARGETS:
    lg, hg = RESULTS[(t, "logistic", "gapped")], RESULTS[(t, "gradient_boosting", "gapped")]
    DEPLOY[t] = "gradient_boosting" if hg["roc_auc_ci"][0] > lg["roc_auc"] else "logistic"
print("Deployed model per target:", DEPLOY)

Out-of-time performance (test = orders from 2018-03-01 on). Random ROC-AUC = 0.5; random PR-AUC = base rate.


,target,model,split,n_train,events_train,n_test,events_test,base_rate_test_pct,mean_pred_test_pct,roc_auc,pr_auc,auc_ci,top_decile_lift
0,went_wrong,logistic,gapped,44815,7453,27987,5001,17.87,16.19,0.628,0.2759,0.620-0.636,1.91
1,went_wrong,gradient_boosting,gapped,44815,7453,27987,5001,17.87,16.63,0.609,0.2633,0.600-0.617,1.84
2,went_wrong,logistic,ungapped,58803,10374,27987,5001,17.87,18.85,0.631,0.2771,0.623-0.639,1.89
3,late,logistic,gapped,44593,3426,27912,2580,9.24,6.38,0.673,0.1756,0.662-0.683,2.42
4,late,gradient_boosting,gapped,44593,3426,27912,2580,9.24,6.68,0.603,0.1418,0.591-0.614,1.93
5,late,logistic,ungapped,58477,4945,27912,2580,9.24,8.82,0.684,0.1790,0.673-0.693,2.43
6,bad_review,logistic,gapped,44815,6160,27987,4026,14.39,13.65,0.630,0.2402,0.621-0.639,2.05
7,bad_review,gradient_boosting,gapped,44815,6160,27987,4026,14.39,13.76,0.618,0.2329,0.609-0.628,2.01
8,bad_review,logistic,ungapped,58803,8623,27987,4026,14.39,15.85,0.633,0.2404,0.622-0.641,2.10
9,canceled,logistic,gapped,44815,222,27987,75,0.27,0.65,0.704,0.0137,0.641-0.771,4.40


Single-feature AUCs on the gapped test set (a model should beat these to be worth having):


,went_wrong,late,bad_review,canceled
distance_km,0.573,0.631,0.558,0.728
promised_days,0.503,0.537,0.522,0.729
seller_prior_late_rate,0.521,0.503,0.526,0.564
seller_prior_bad_review_rate,0.556,0.509,0.565,0.598


Deployed model per target: {'went_wrong': 'logistic', 'late': 'logistic', 'bad_review': 'logistic', 'canceled': 'logistic'}


In [11]:
print("Gapped out-of-time deciles for the deployed composite model (decile 1 = highest predicted risk):")
display(RESULTS[("went_wrong", DEPLOY["went_wrong"], "gapped")]["deciles"])

# Historical outcome per band (bands = top 10% / 50-90% / bottom 50% of the gapped model's TRAINING scores), measured out-of-time
BAND_OUTCOMES = {}
for t in TARGETS:
    r = RESULTS[(t, DEPLOY[t], "gapped")]
    q50, q90 = np.quantile(r["p_train"], [.5, .9])
    band = np.where(r["p_test"] >= q90, "High", np.where(r["p_test"] >= q50, "Medium", "Low"))
    rows = []
    for b in ["High", "Medium", "Low"]:
        y = r["y_test"][band == b]; k, n = int(y.sum()), len(y); lo, hi = wilson(k, n)
        rows.append(dict(band=b, orders=n, events=k, rate_pct=round(100 * k / max(n, 1), 2), ci_low=round(lo, 2), ci_high=round(hi, 2)))
    BAND_OUTCOMES[t] = rows
    print(f"\n{t} ({DEPLOY[t]}): out-of-time rate by band (test base {r['base_rate_test_pct']}%)")
    display(pd.DataFrame(rows))

Gapped out-of-time deciles for the deployed composite model (decile 1 = highest predicted risk):


,decile,orders,events,mean_pred_pct,actual_pct,lift
0,1,2799,954,34.12,34.08,1.91
1,2,2799,660,23.00,23.58,1.32
2,3,2798,655,19.29,23.41,1.31
3,4,2799,507,16.82,18.11,1.01
4,5,2798,473,14.92,16.90,0.95
5,6,2799,422,13.36,15.08,0.84
6,7,2799,376,12.13,13.43,0.75
7,8,2798,335,10.96,11.97,0.67
8,9,2799,334,9.65,11.93,0.67
9,10,2799,285,7.67,10.18,0.57



went_wrong (logistic): out-of-time rate by band (test base 17.87%)


,band,orders,events,rate_pct,ci_low,ci_high
0,High,2594,901,34.73,32.93,36.59
1,Medium,10117,2132,21.07,20.29,21.88
2,Low,15276,1968,12.88,12.36,13.42



late (logistic): out-of-time rate by band (test base 9.24%)


,band,orders,events,rate_pct,ci_low,ci_high
0,High,1272,300,23.58,21.33,26.00
1,Medium,9647,1308,13.56,12.89,14.26
2,Low,16993,972,5.72,5.38,6.08



bad_review (logistic): out-of-time rate by band (test base 14.39%)


,band,orders,events,rate_pct,ci_low,ci_high
0,High,2968,859,28.94,27.34,30.60
1,Medium,9868,1642,16.64,15.92,17.39
2,Low,15151,1525,10.07,9.60,10.55



canceled (logistic): out-of-time rate by band (test base 0.27%)


,band,orders,events,rate_pct,ci_low,ci_high
0,High,3888,37,0.95,0.69,1.31
1,Medium,13163,22,0.17,0.11,0.25
2,Low,10936,16,0.15,0.09,0.24


## 6. Experiment design: how many orders/customers per arm?
Two-sided α = 0.05, 80% power, normal approximation. Base rates come from this run. This is why the experiment
readout uses a **fast, frequent outcome** (bad review, went wrong) as primary and 180-day repeat only as secondary.

In [12]:
from statistics import NormalDist
def n_per_arm(p0, rel_change, alpha=0.05, power=0.80):
    p1 = p0 * (1 + rel_change); za, zb = NormalDist().inv_cdf(1 - alpha / 2), NormalDist().inv_cdf(power); pb = (p0 + p1) / 2
    return math.ceil((za * math.sqrt(2 * pb * (1 - pb)) + zb * math.sqrt(p0 * (1 - p0) + p1 * (1 - p1))) ** 2 / (p1 - p0) ** 2)

base = {"bad_review": pop.bad_review.mean(), "went_wrong": pop.went_wrong.mean(), "repeat_180d": fc.repeat_180d.mean()}
plan = pd.DataFrame([{"outcome": k, "base_rate_pct": round(v * 100, 2), "relative_change": rc, "target_rate_pct": round(v * (1 + rc) * 100, 2),
                      "n_per_arm": n_per_arm(v, rc)} for k, v in base.items() for rc in (-0.15, -0.30, 0.30) if 0 < v * (1 + rc) < 1])
monthly = o[o.order_purchase_timestamp >= OBS_END - pd.Timedelta(days=365)].groupby(o.order_purchase_timestamp.dt.to_period("M")).size().median()
plan["months_at_typical_volume"] = (2 * plan.n_per_arm / monthly).round(1)
print(f"Typical monthly order volume (last 12 months, median): {monthly:,.0f}")
display(plan)

Typical monthly order volume (last 12 months, median): 6,402


,outcome,base_rate_pct,relative_change,target_rate_pct,n_per_arm,months_at_typical_volume
0,bad_review,14.57,-0.15,12.39,3830,1.2
1,bad_review,14.57,-0.30,10.20,891,0.3
2,bad_review,14.57,0.30,18.95,1145,0.4
3,went_wrong,17.72,-0.15,15.06,3045,1.0
4,went_wrong,17.72,-0.30,12.40,710,0.2
5,went_wrong,17.72,0.30,23.03,901,0.3
6,repeat_180d,1.95,-0.15,1.66,32526,10.2
7,repeat_180d,1.95,-0.30,1.36,7483,2.3
8,repeat_180d,1.95,0.30,2.53,10064,3.1


## 7. Fit the deployed models on all resolved orders, export the artifact and scored orders

In [13]:
VERSION = "olist-order-risk-v3-" + pd.Timestamp.now().strftime("%Y%m%d")
targets_art, scored = {}, df[["order_id", "customer_unique_id", "seller_id", "order_purchase_timestamp", "order_status", "in_population"] + TARGETS + FEATURES].copy()
scored["split"] = np.where(~df.in_population, "unresolved/excluded", np.where(df.order_purchase_timestamp >= TEST_START, "test", "train"))
has_x = df.order_id.isin(o.loc[o.has_items, "order_id"])
for t in TARGETS:
    d = data[t]; pipe = FACTORIES[DEPLOY[t]]().fit(d[FEATURES], d[t].astype(int))
    p_fit = pipe.predict_proba(d[FEATURES])[:, 1]
    q = {"p50": float(np.quantile(p_fit, .5)), "p90": float(np.quantile(p_fit, .9))}
    r = RESULTS[(t, DEPLOY[t], "gapped")]
    targets_art[t] = {"pipeline": pipe, "model_type": DEPLOY[t], "base_rate": float(d[t].mean()), "score_quantiles": q,
                      "n_train": int(len(d)), "n_events": int(d[t].sum()),
                      "metrics": {"oot_gapped": {k: r[k] for k in ("roc_auc", "roc_auc_ci", "pr_auc", "base_rate_test_pct", "mean_pred_test_pct", "n_test", "events_test", "single_feature_auc")}
                                  | {"deciles": r["deciles"].to_dict("records")},
                                  "oot_ungapped_logistic_auc": RESULTS[(t, "logistic", "ungapped")]["roc_auc"],
                                  "comparison": perf[(perf.target == t)].drop(columns=["target"]).to_dict("records")},
                      "band_outcomes_oot": BAND_OUTCOMES[t]}
    p = np.full(len(df), np.nan); p[has_x.values] = pipe.predict_proba(df.loc[has_x, FEATURES])[:, 1]
    scored[f"p_{t}"] = p
    scored[f"band_{t}"] = np.where(np.isnan(p), None, np.where(p >= q["p90"], "High", np.where(p >= q["p50"], "Medium", "Low")))

# Seller track record as of the end of the data (what the app uses to score a NEW order)
now = OBS_END + pd.Timedelta(seconds=1)
sq = sellers[["seller_id"]].assign(order_id=sellers.seller_id, ts=now)
st = sellers.rename(columns={"seller_zip_code_prefix": "seller_zip"})[["seller_id", "seller_zip", "seller_state"]]
for name, prefix in [("orders", "seller_orders"), ("late", "seller_late"), ("bad", "seller_bad"), ("cancel", "seller_cancel")]:
    pc = prior_counts(EVENTS[name], sq, "ts", prefix).rename(columns={"order_id": "seller_id"})
    st = st.merge(pc, on="seller_id", how="left", validate="one_to_one")
st = st.rename(columns={"seller_orders_n": "seller_prior_orders"}).drop(columns="seller_orders_k").fillna({"seller_state": "missing"})
cats = sorted(df.loc[df.in_population, "category"].value_counts().loc[lambda s: s >= 20].index.tolist())

artifact = {
    "model_version": VERSION, "kind": "order_risk", "sklearn_version": sklearn.__version__, "pandas_version": pd.__version__,
    "features": FEATURES, "num": NUM, "cat": CAT, "targets": targets_art, "deployed": DEPLOY,
    "global_rates": GLOBAL_RATES, "shrink_a": SHRINK_A,
    "lookups": {"zip_centroids": {int(k): [v[0], v[1]] for k, v in ZIP_CENTROIDS.items()},
                "seller_stats": st.to_dict("list"), "categories": cats,
                "states": sorted(df.loc[df.in_population, "customer_state"].unique().tolist()),
                "payment_types": sorted(df.loc[df.in_population, "payment_type"].unique().tolist())},
    "definitions": {"late": "delivered after the promised date, or never delivered once the promise date passed",
                    "bad_review": "lowest review score on the order <= 2", "canceled": "status canceled or unavailable",
                    "went_wrong": "late OR bad_review OR canceled"},
    "scoring_time": "at purchase - only facts known when the order is placed; seller history is point-in-time",
    "data_end": str(OBS_END), "test_start": str(TEST_START.date()), "resolve_days": RESOLVE_DAYS,
    "caveats": ["Observational: a high score says an order is at risk, not what would fix it.",
                "Seller history in the app is as of the end of the data (2018); refresh it with current data before real use.",
                "Trained on one marketplace and period (incl. any 2018 logistics disruptions); re-validate before reuse."],
}
joblib.dump(artifact, OUT_DIR / "order_risk_model.joblib")
chk = joblib.load(OUT_DIR / "order_risk_model.joblib")
for t in TARGETS:
    a = chk["targets"][t]["pipeline"].predict_proba(df.loc[has_x, FEATURES].head(200))[:, 1]
    assert np.allclose(a, scored.loc[has_x, f"p_{t}"].head(200).values), f"reload mismatch for {t}"

# Train/serve check: rebuild 400 orders the way the app does (lookups read back from the saved artifact)
zc_app = {int(k): tuple(v) for k, v in chk["lookups"]["zip_centroids"].items()}
idx = rng.choice(np.flatnonzero(has_x.values), 400, replace=False)
Xa = build_frame(raw.iloc[idx][INPUT_COLS], zc_app, chk["global_rates"])
Xb = X_all.iloc[idx].reset_index(drop=True)
diff = [f for f in FEATURES if not (Xa[f].equals(Xb[f]) or np.allclose(pd.to_numeric(Xa[f], errors="coerce").fillna(-9e9), pd.to_numeric(Xb[f], errors="coerce").fillna(-9e9))
                                    if f in NUM else (Xa[f].astype(str) == Xb[f].astype(str)).all())]
assert not diff, f"train/serve skew in {diff}"
scored.to_csv(OUT_DIR / "orders_scored.csv", index=False)
print(f"Saved order_risk_model.joblib ({VERSION}, scikit-learn {sklearn.__version__}) | reload OK | train/serve check OK on 400 orders x {len(FEATURES)} features")
print(f"Saved orders_scored.csv ({len(scored):,} orders; {int(has_x.sum()):,} scored)")

Saved order_risk_model.joblib (olist-order-risk-v3-20261002, scikit-learn 1.6.1) | reload OK | train/serve check OK on 400 orders x 27 features
Saved orders_scored.csv (99,441 orders; 98,666 scored)


## 8. Auto-generated summary (numbers come from this run)

In [14]:
ws = summ.set_index("target")
print(f"1) Population: {len(pop):,} resolved orders. Went wrong: {ws.loc['went_wrong','rate_pct']}% "
      f"(late {ws.loc['late','rate_pct']}%, bad review {ws.loc['bad_review','rate_pct']}%, canceled {ws.loc['canceled','rate_pct']}%).")
for t in TARGETS:
    r = RESULTS[(t, DEPLOY[t], "gapped")]; best_single = max(r["single_feature_auc"].items(), key=lambda kv: kv[1])
    print(f"2) {t:<10} {DEPLOY[t]:<17} gapped OOT AUC {r['roc_auc']:.3f} ({r['roc_auc_ci'][0]:.3f}-{r['roc_auc_ci'][1]:.3f}); "
          f"top decile {r['deciles'].actual_pct.iloc[0]}% vs base {r['base_rate_test_pct']}% ({r['deciles'].lift.iloc[0]}x); "
          f"best single feature {best_single[0]} {best_single[1]:.3f}")
b = pd.DataFrame(BAND_OUTCOMES["went_wrong"]).set_index("band")
sep = "separate" if b.loc["High", "ci_low"] > b.loc["Low", "ci_high"] else "OVERLAP"
print(f"3) Composite bands out-of-time: High {b.loc['High','rate_pct']}% ({b.loc['High','ci_low']}-{b.loc['High','ci_high']}) vs "
      f"Low {b.loc['Low','rate_pct']}% ({b.loc['Low','ci_low']}-{b.loc['Low','ci_high']}) -> intervals {sep}.")
r = RESULTS[("went_wrong", DEPLOY["went_wrong"], "gapped")]
print(f"4) Calibration on the test period: mean predicted {r['mean_pred_test_pct']}% vs actual {r['base_rate_test_pct']}% "
      "(a gap here means the risk level shifted over time; use bands, and recalibrate on recent data before real use).")
print(f"5) Retention link: repeat {ww.loc['yes','repeat_pct']}% after a first order that went wrong vs {ww.loc['no','repeat_pct']}% otherwise (association, not effect).")
print("6) Not established: that warning customers, expediting or flagging sellers reduces failures or raises repeat. That is what the experiment measures.")

1) Population: 86,790 resolved orders. Went wrong: 17.72% (late 8.71%, bad review 14.57%, canceled 0.46%).
2) went_wrong logistic          gapped OOT AUC 0.628 (0.620-0.636); top decile 34.08% vs base 17.87% (1.91x); best single feature distance_km 0.573
2) late       logistic          gapped OOT AUC 0.673 (0.662-0.683); top decile 22.39% vs base 9.24% (2.42x); best single feature distance_km 0.631
2) bad_review logistic          gapped OOT AUC 0.630 (0.621-0.639); top decile 29.51% vs base 14.39% (2.05x); best single feature seller_prior_bad_review_rate 0.565
2) canceled   logistic          gapped OOT AUC 0.704 (0.641-0.771); top decile 1.18% vs base 0.27% (4.4x); best single feature promised_days 0.729
3) Composite bands out-of-time: High 34.73% (32.93-36.59) vs Low 12.88% (12.36-13.42) -> intervals separate.
4) Calibration on the test period: mean predicted 16.19% vs actual 17.87% (a gap here means the risk level shifted over time; use bands, and recalibrate on recent data before re